In [ ]:
import numpy as np 
import pandas as pd 
import os
import sys
import requests
import shutil
import torch
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import timm
import kagglehub


print("Installing dependencies and cloning repositories...")
!pip install -q timm yacs iopath kagglehub
!pip install -q git+https://github.com/fra31/auto-attack.git

os.chdir('/content')
if os.path.exists('/content/SAR'): shutil.rmtree('/content/SAR')
if os.path.exists('/content/test-time-adaptation'): shutil.rmtree('/content/test-time-adaptation')

!git clone https://github.com/mr-eggplant/SAR.git /content/SAR
!git clone --depth 1 https://github.com/mariodoebler/test-time-adaptation.git /content/test-time-adaptation


sys.path.append('/content/SAR')
sys.path.insert(0, '/content/test-time-adaptation/classification')


import sar
def stable_softmax_entropy(x):
    return -(x.softmax(1) * x.log_softmax(1)).sum(1)
sar.softmax_entropy = stable_softmax_entropy


import types
import robustbench
try:
    import robustbench.loaders as rb_loaders
except ImportError:
    rb_loaders = types.ModuleType('loaders')
    robustbench.loaders = rb_loaders
    sys.modules['robustbench.loaders'] = rb_loaders

class DummyDataset(torch.utils.data.Dataset):
    def __init__(self, *args, **kwargs): pass
    def __len__(self): return 0
rb_loaders.CustomCifarDataset = DummyDataset
rb_loaders.CustomImageFolder = datasets.ImageFolder


In [ ]:
data_path = '/kaggle/input/datasets/sampadramkumar/gaussian-noise/kaggle/working/gaussian_noise_5'

# ROID

In [ ]:
import os
import sys
import requests
import shutil
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
from tqdm import tqdm
import timm


print("Installing dependencies...")
!pip install -q timm yacs iopath kagglehub
!pip install -q git+https://github.com/fra31/auto-attack.git


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ROID(nn.Module):
    def __init__(self, model, lr=2.5e-4, alpha=0.99, beta=0.9, tau=1/3):
        super().__init__()
        self.model = model

        
        self.model_0 = copy.deepcopy(model)
        self.model_0.requires_grad_(False)
        self.model_0.eval()

        params_to_train = []
        for m in self.model.modules():
            
            if isinstance(m, nn.LayerNorm):
                m.requires_grad_(True)
                params_to_train.append(m.weight)
                params_to_train.append(m.bias)
            else:
                m.requires_grad_(False)

        
        self.optimizer = torch.optim.SGD(params_to_train, lr=lr, momentum=0.9)
        self.alpha = alpha
        self.beta = beta
        self.tau = tau
        self.tendency = None

    def slr_loss(self, probs):
        probs = torch.clamp(probs, min=1e-5, max=1.0 - 1e-5)
        neg_probs = 1.0 - probs
        slr = probs * torch.log(probs / neg_probs)
        return -slr.sum(dim=1)

    def sce_loss(self, p1, p2):
        p1 = torch.clamp(p1, min=1e-5, max=1.0 - 1e-5)
        p2 = torch.clamp(p2, min=1e-5, max=1.0 - 1e-5)
        loss1 = -(p1 * torch.log(p2)).sum(dim=1)
        loss2 = -(p2 * torch.log(p1)).sum(dim=1)
        return 0.5 * (loss1 + loss2)

    def adapt(self, x):
        self.model.train()
        
        x_aug = transforms.functional.hflip(x)

        outputs = self.model(x)
        outputs_aug = self.model(x_aug)

        probs = outputs.softmax(dim=1)
        probs_aug = outputs_aug.softmax(dim=1)
        
        batch_mean_probs = probs.mean(dim=0).detach()
        if self.tendency is None:
            self.tendency = batch_mean_probs
        else:
            self.tendency = self.beta * self.tendency + (1 - self.beta) * batch_mean_probs

        sim = F.cosine_similarity(probs.detach(), self.tendency.unsqueeze(0), dim=1)
        w_div = 1.0 - sim
        ent = -(probs.detach() * torch.log(probs.detach() + 1e-6)).sum(dim=1)
        w_cert = -ent

        def normalize_01(w):
            w_min, w_max = w.min(), w.max()
            if w_max > w_min: return (w - w_min) / (w_max - w_min)
            return w

        w_final = torch.exp((normalize_01(w_div) * normalize_01(w_cert)) / self.tau)
        w_final = w_final / (w_final.mean() + 1e-8)

        
        loss_slr = self.slr_loss(probs)
        loss_cons = self.sce_loss(probs, probs_aug)
        loss = (w_final * loss_slr).mean() + loss_cons.mean()

        if loss.requires_grad:
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

        
        with torch.no_grad():
            for p, p0 in zip(self.model.parameters(), self.model_0.parameters()):
                if p.requires_grad:
                    p.data = self.alpha * p.data + (1 - self.alpha) * p0.data

        self.model.eval()

        
        with torch.no_grad():
            final_outputs = self.model(x)
            final_probs = final_outputs.softmax(dim=1)
            p_hat = final_probs.mean(dim=0)
            N_b, N_c = final_probs.size(0), final_probs.size(1)

            gamma = max(1.0/N_b, 1.0/N_c) / (p_hat.max() + 1e-6)
            p_bar = (p_hat + gamma) / (1 + gamma * N_c)

            corrected_probs = final_probs * (p_bar * N_c).unsqueeze(0)
            corrected_probs = corrected_probs / corrected_probs.sum(dim=1, keepdim=True)

        return corrected_probs




print("Loading ViT-Base (vit_base_patch16_224)...")

model = timm.create_model('vit_base_patch16_224', pretrained=True).to(device)


timm_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]) 
])


print(f"Loading data from: {data_path}")
dataset = datasets.ImageFolder(root=data_path, transform=timm_transform)


url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
mapping = requests.get(url).json()
wnid_to_idx = {v[0]: int(k) for k, v in mapping.items()}

new_samples = [(p, wnid_to_idx[os.path.basename(os.path.dirname(p))])
               for p, _ in dataset.samples if os.path.basename(os.path.dirname(p)) in wnid_to_idx]
dataset.samples = new_samples
dataset.targets = [s[1] for s in new_samples]

dataloader = DataLoader(dataset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)


print("\nRunning Baseline (No Adaptation)...")
model.eval()
base_correct, base_total = 0, 0
with torch.no_grad():
    for i, (images, labels) in enumerate(dataloader):
        if i >= 10: break
        preds = model(images.to(device)).argmax(1)
        base_correct += (preds == labels.to(device)).sum().item()
        base_total += labels.size(0)

print(f"Baseline Accuracy: {base_correct / base_total * 100:.2f}%")

roid = ROID(model, lr=2.5e-4).to(device)
correct, total = 0, 0

print("\nStarting Online TTA (ROID on ViT)...")
for images, labels in tqdm(dataloader):
    images, labels = images.to(device), labels.to(device)
    outputs = roid.adapt(images)
    preds = outputs.argmax(1)
    correct += (preds == labels).sum().item()
    total += labels.size(0)

print(f"\nFinal TTA Accuracy: {correct / total * 100:.2f}%")